In [4]:
import os
data_path = "../data/animals"
data_to_process = []
for path, foldernames, filenames in os.walk(data_path):
    for filename in filenames:
        if filename.startswith("."):
            continue
        file_path = os.path.join(path, filename)
        data_to_process.append({'label': file_path.split("/")[-2], 'path': file_path})
        


In [7]:
import json
import torch
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

def generate_captions_and_save(dataset, output_file):
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print("Uzywanie urzadzenia:", device)

    print("Pobieranie danych...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
    model.to(device)
    model.eval()

    print("Generowanie opisów...")
    for item in tqdm(dataset, desc="Generowanie opisów"):
        image_path = item['path']
        try:
            raw_image = Image.open(image_path).convert("RGB")
            inputs = processor(raw_image, return_tensors="pt").to(device)

            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=20)

            caption = processor.decode(out[0], skip_special_tokens=True)

            item['caption'] = caption
        except Exception as e:
            print("Błąd podczas generowania opisu dla obrazu:", image_path)
            item['caption'] = "Error generating caption"

    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=4, ensure_ascii=False)

    print("Zakończono generowanie opisów.")




In [8]:
plik_wynikowy = "dataset_captions.json"
generate_captions_and_save(data_to_process, plik_wynikowy)

Uzywanie urzadzenia: mps
Pobieranie danych...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 49622.46it/s]


Generowanie opisów...


Generowanie opisów: 100%|██████████| 5400/5400 [25:35<00:00,  3.52it/s]

Zakończono generowanie opisów.
